# Estrategia "mínimo histórico" — de punta a punta

Prototipar la lógica con el builder, evaluarla sobre la serie **real** de la app, correrla
*como la corre la app* y exportar el JSON para importarlo desde la pantalla de Estrategias.

Lo exportable es la **lógica**, no las fechas: la misma estrategia sobre otro ticker da otros
momentos de entrada y salida.

In [ ]:
from app.lab import Estrategia, ind, precio, entre, todas, alguna, negar
from app.lab import barras_de_db, barras_de_dataframe, correr_como_la_app, comparar_con_mascara

TICKER = "AAPL"        # un ticker de tu cartera o watchlist
VARIANTE = "subyacente"  # "local" | "subyacente" (USD)
DESDE = "2015-01-01"

In [ ]:
canal = ind.EXTREMOS(ventana=0)   # 0 = histórico acumulado

est = (Estrategia("Mínimo histórico")
       .comprar(canal.dist_min_pct <= 1.0)   # a <=1% del mínimo histórico
       .vender(canal.dist_max_pct >= -1.0)   # a <=1% del máximo histórico
       .riesgo(stop_loss_pct=20)
       .ejecucion(comision_pct=0.6, precio_ejecucion="apertura_siguiente", demora_barras=1))

print(est.a_json())

In [ ]:
barras = barras_de_db(TICKER, desde=DESDE, variante=VARIANTE)
df = est.dataframe(barras)   # OHLCV + una columna por salida de indicador + entrada/salida
df.tail()

In [ ]:
res = est.backtest(barras)   # ResultadoLab: .metricas, .operaciones, .senales, .equity
res

## Verificación: correrla *como la app*

`correr_como_la_app` llama a `estrategias_analytics.ejecutar_backtest` — el mismo camino de
warm-up, `indice_inicio` y `max_barras` que la pantalla. **Las métricas de acá tienen que
coincidir con las del backtest en la app** después de importar el JSON.

In [ ]:
salida = correr_como_la_app(TICKER, est, desde=DESDE, variante=VARIANTE)
salida["metricas"]

In [ ]:
import pandas as pd
pd.DataFrame(salida["operaciones"])

In [ ]:
est.exportar("estrategias/estrategia-minimo-historico.json")

## Variante estricta y por qué no se usa

La forma "literal" sería `precio.cierre <= canal.minimo`. Casi nunca dispara: como el extremo
incluye la barra actual, con velas reales el cierre sólo iguala al mínimo si cerró exacto en el
piso del día. Por eso el ejemplo usa la tolerancia porcentual `dist_min_pct <= 1`. (Para "hoy
hizo máximo nuevo" el modismo correcto es `canal.maximo.subiendo()`: la serie `maximo` crece
exactamente el día del máximo nuevo.)

In [ ]:
est_estricta = Estrategia("Mínimo histórico (estricta)").comprar(precio.cierre <= canal.minimo)
est_estricta.backtest(barras).metricas

## Cerrar el círculo: `comparar_con_mascara`

El builder deliberadamente **no** traduce máscaras de pandas al DSL (una máscara es un vector de
resultados, no una expresión; y `df.cierre.shift(-1)` abriría la puerta al lookahead). El camino
inverso sí: prototipás libre en pandas y después *verificás* que tu traducción al builder
coincide. `comparar_con_mascara` devuelve las barras donde difieren.

In [ ]:
mascara = df["cierre"] <= df["cierre"].cummin() * 1.01   # "a <=1% del mínimo acumulado"
comparar_con_mascara(est, barras, mascara)   # idealmente vacío o casi